# AbstractComm: Discrete Multi-Agent Communication Framework

## 1. Executive Summary & Overview
This notebook implements the **AbstractComm** framework for discrete multi-agent communication between Large Language Models (LLMs). The architecture enables a **Researcher Agent ($M_A$)** and a **Reader Agent ($M_B$)** to solve multi-hop question answering tasks (e.g., HotpotQA) by communicating through a bottlenecked, learned discrete codebook.

### Core Principles
1. **Information Bottleneck:** $M_A$ compresses large multi-hop Wikipedia contexts into a small sequence of discrete abstract tokens ($K=8$ tokens from a vocabulary of $|V|=26$).
2. **Continuous Relaxation:** Uses **Gumbel-Softmax** sampling to allow end-to-end gradient backpropagation through discrete bottleneck tokens during Supervised Fine-Tuning (SFT).
3. **Channel Robustness:** Trains $M_B$ under synthetic channel noise to ensure message fault tolerance.
4. **Policy Optimization:** Uses **Group Relative Policy Optimization (GRPO)** to optimize answer accuracy while rewarding message diversity and penalizing redundancy.

## Step 1: Environment Setup & Dependency Installation

We install the required libraries for quantizing 3B LLMs (`bitsandbytes`), parameter-efficient fine-tuning (`peft`), and dataset management (`datasets`). Compatible package bounds are pinned to avoid PyArrow ABI conflicts.

In [1]:
!pip install -q -U torchao peft transformers datasets trl accelerate sentencepiece huggingface_hub bitsandbytes

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.optim as optim
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, PeftModel
from datasets import load_dataset
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

## Step 2: Workspace Setup & Authentication

Configures Kaggle output directories, Hugging Face caching, and authenticates via Kaggle User Secrets (`HF_TOKEN`) for model hub checkpointing.

In [2]:
# Setup Kaggle Cache Paths
os.environ["HF_HOME"] = "/kaggle/working/cache/huggingface"
os.environ["DATASETS_CACHE"] = "/kaggle/working/cache/datasets"
OUTPUT_DIR = "/kaggle/working/abstractcomm-qwen3b"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Authenticate with Hugging Face via Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Successfully authenticated with Hugging Face Hub.")
except Exception as e:
    print(f"HF_TOKEN not found in Kaggle Secrets. Manual login required for hub upload: {e}")

## Step 3: Global Hyperparameters & Protocol Specifications

Defines protocol constraints and loss hyperparameter weights derived from the AbstractComm proposal:

- **Vocabulary Size ($\vert{}V\vert{}=26$):** Defines the codebook size ($V = \{\text{<TOKEN\_A>}, \dots, \text{<TOKEN\_Z>}\}$).
- **Message Length ($K=8$):** Fixed discrete bandwidth constraint for agent communication.
- **$\lambda_1 = 0.01$ (Utilization Loss Weight):** Encourages uniform codebook usage to prevent collapse.
- **$\lambda_2 = 0.05$ (Separation Loss Weight):** Forces distinct contexts to produce distinct abstract representations ($d_{\text{cos}} \ge D_{\text{min}} = 0.3$).
- **$\beta_1 = 0.1, \beta_2 = 0.05$ (GRPO Policy Weights):** Diversity reward weight and compression penalty weight.

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VOCAB_SIZE = 26      # V = 26 abstract tokens (<TOKEN_A> to <TOKEN_Z>)
MESSAGE_LEN = 8      # K = 8 fixed token bandwidth
BETA1 = 0.1          # GRPO diversity reward weight
BETA2 = 0.05         # GRPO compression penalty weight
D_MIN = 0.3          # Minimum cosine distance for separation loss
print(f"Device Name: {torch.cuda.get_device_name(0)}")

## Step 4: Dataset Loading & Formatting (HotpotQA)

HotpotQA multi-hop reasoning contexts are preprocessed into structured prompts for $M_A$ and target answers for $M_B$:

- **Researcher Input Prompt ($x$):** $\text{"Context: "} + \text{Wikipedia Passages} + \text{" Question: "} + q + \text{" Summarize reasoning for Reader:"}$
- **Reader Target Output ($y$):** Ground-truth target answer string $a^*$.
- **Dataset Subsampling:** Subsampled to $500$ instances to ensure fast iteration and execution within Kaggle session time limits.

In [4]:
dataset = load_dataset("hotpotqa/hotpot_qa", "distractor", cache_dir=os.environ["DATASETS_CACHE"])

def preprocess_hotpotqa(example):
    context = ""
    for title, sentences in zip(example['context']['title'], example['context']['sentences']):
        context += f"Title: {title}\n" + " ".join(sentences) + "\n\n"
    
    question = example['question']
    answer = example['answer']
    
    researcher_prompt = f"Context:\n{context}\nQuestion: {question}\nSummarize reasoning for Reader:"
    reader_target = answer
    
    return {
        "researcher_input": researcher_prompt,
        "reader_target": reader_target
    }

subsampled_dataset = dataset["train"].select(range(500))

processed_train = subsampled_dataset.map(
    preprocess_hotpotqa,
    remove_columns=dataset['train'].column_names,
    desc="Preprocessing HotpotQA (Subsampled 500)"
)

# Store processed dataset
processed_dataset = {
    "train": processed_train
}

print(f"Dataset successfully subsampled and preprocessed: {len(processed_dataset['train'])} samples.")

## Step 5: Dual-Agent Architecture & Continuous Relaxation

### Mathematical Formulation
1. **Hidden Representation:** Researcher $M_A$ processes context sequence $x$ to extract final hidden state $h_A \in \mathbb{R}^{d}$.
2. **Codebook Projection:** Linear projection head produces logits $Z \in \mathbb{R}^{K \times \vert{}V\vert{}}$:
   $$Z = W_{\text{codebook}} h_A + b$$
3. **Gumbel-Softmax Continuous Relaxation:** To allow backpropagation through discrete samples, soft categorical samples $m \in \mathbb{R}^{K \times \vert{}V\vert{}}$ are drawn using temperature $\tau$ and i.i.d. Gumbel noise $g_k,v \sim \text{Gumbel}(0, 1)$:
   $$m_{k,v} = \frac{\exp((Z_{k,v} + g_{k,v}) / \tau)}{\sum_{j=1}^{\vert{}V\vert{}} \exp((Z_{k,j} + g_{k,j}) / \tau)}$$
4. **Embedding Lookup:** Continuous message matrix $m$ is multiplied by codebook embedding weight matrix $E_{\text{codebook}} \in \mathbb{R}^{\vert{}V\vert{} \times d}$ to produce continuous input embeddings $e_m = m \cdot E_{\text{codebook}}$ for Reader $M_B$.
5. **Quantization & LoRA:** Both models use 4-bit NormalFloat (`nf4`) quantization with Low-Rank Adaptation ($r=8, \alpha=16$) to optimize memory on Kaggle GPUs.

In [5]:
from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

class HomogeneousAbstractComm(nn.Module):
    def __init__(self, model_id="Qwen/Qwen2.5-3B-Instruct"):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, 
            trust_remote_code=True,
            cache_dir=os.environ["HF_HOME"]
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # 1. Configure 4-bit Quantization
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True
        )

        # 2. Load Base Models in 4-bit
        self.researcher = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=bnb_config,
            device_map=DEVICE, 
            cache_dir=os.environ["HF_HOME"]
        )
        self.reader = AutoModelForCausalLM.from_pretrained(
            model_id, 
            quantization_config=bnb_config,
            device_map=DEVICE, 
            cache_dir=os.environ["HF_HOME"]
        )

        # 3. Enable k-bit training
        self.researcher = prepare_model_for_kbit_training(self.researcher)
        self.reader = prepare_model_for_kbit_training(self.reader)
        
        # 4. Attach LoRA Adapters
        lora_config = LoraConfig(
            task_type="CAUSAL_LM",
            r=8, 
            lora_alpha=16, 
            target_modules=["q_proj", "v_proj"]
        )
        self.researcher = get_peft_model(self.researcher, lora_config)
        self.reader = get_peft_model(self.reader, lora_config)

        hidden_dim = self.researcher.config.hidden_size

        # Abstract Token Projection Head & Shared Codebook Table
        self.codebook_head = nn.Linear(hidden_dim, MESSAGE_LEN * VOCAB_SIZE).to(DEVICE)
        self.codebook_embeddings = nn.Embedding(VOCAB_SIZE, hidden_dim).to(DEVICE)
        nn.init.normal_(self.codebook_embeddings.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask, target_ids=None, tau=1.0, hard=True, noise_rate=0.0):
        r_out = self.researcher(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        h_A = r_out.hidden_states[-1][:, -1, :]
        
        logits = self.codebook_head(h_A).view(-1, MESSAGE_LEN, VOCAB_SIZE)
        
        if noise_rate > 0.0 and self.training:
            noise_mask = (torch.rand_like(logits[..., 0]) < noise_rate).unsqueeze(-1)
            random_logits = torch.randn_like(logits)
            logits = torch.where(noise_mask, random_logits, logits)

        message_soft = F.gumbel_softmax(logits, tau=tau, hard=hard)
        m_embeds = torch.matmul(message_soft, self.codebook_embeddings.weight.to(message_soft.dtype))

        if target_ids is not None:
            r_target_embeds = self.reader.get_input_embeddings()(target_ids)
            combined_embeds = torch.cat([m_embeds, r_target_embeds], dim=1)
            
            reader_out = self.reader(inputs_embeds=combined_embeds)
            
            logits_pred = reader_out.logits[:, MESSAGE_LEN:-1, :]
            labels = target_ids[:, 1:]
            loss_answer = F.cross_entropy(logits_pred.reshape(-1, logits_pred.size(-1)), labels.reshape(-1))
            return loss_answer, logits, message_soft
        
        return logits, message_soft

system = HomogeneousAbstractComm()
optimizer = optim.AdamW(system.parameters(), lr=1e-4)

## Step 6: Multi-Component Loss Functions

### Total Joint SFT Objective
$$\mathcal{L}_{\text{SFT}} = \mathcal{L}_{\text{answer}} + \lambda_1 \mathcal{L}_{\text{util}} + \lambda_2 \mathcal{L}_{\text{sep}}$$

1. **Answer Cross-Entropy Loss ($\mathcal{L}_{\text{answer}}$):** Standard token-level negative log-likelihood over ground-truth target answer $y$.
2. **Codebook Utilization Loss ($\mathcal{L}_{\text{util}}$):** Prevents **codebook collapse** by forcing average token output distribution $\bar{p}$ toward a uniform distribution over $\vert{}V\vert{}$ tokens:
   $$\mathcal{L}_{\text{util}} = \log \vert{}V\vert{} - H(\bar{p}) = \log \vert{}V\vert{} + \sum_{v=1}^{\vert{}V\vert{}} \bar{p}_v \log \bar{p}_v$$
3. **Separation Loss ($\mathcal{L}_{\text{sep}}$):** Ensures distinct input contexts produce distinct message representations ($d_{\text{cos}} \ge D_{\text{min}} = 0.3$):
   $$\mathcal{L}_{\text{sep}} = \frac{1}{\binom{B}{2}} \sum_{i < j} \max\left(0, D_{\text{min}} - (1 - \cos(z_i, z_j))\right)$$

In [6]:
def compute_joint_sft_loss(loss_answer, logits, message_soft, lambda1=0.01, lambda2=0.05):
    p_bar = F.softmax(logits, dim=-1).mean(dim=[0, 1])
    loss_util = torch.log(torch.tensor(VOCAB_SIZE, device=DEVICE)) + torch.sum(p_bar * torch.log(p_bar + 1e-8))

    z = message_soft.mean(dim=1)
    z_norm = F.normalize(z, p=2, dim=-1)
    cos_sim = torch.mm(z_norm, z_norm.t())
    
    batch_size = z.size(0)
    if batch_size > 1:
        mask = ~torch.eye(batch_size, dtype=torch.bool, device=DEVICE)
        d_cos = 1.0 - cos_sim[mask]
        loss_sep = torch.mean(F.relu(D_MIN - d_cos))
    else:
        loss_sep = torch.tensor(0.0, device=DEVICE)

    total_loss = loss_answer + (lambda1 * loss_util) + (lambda2 * loss_sep)
    return total_loss, loss_util, loss_sep

## Step 7: DataLoader & Batch Configuration

Instantiates `DataLoader` with `batch_size = 8` and `gradient_accumulation_steps = 2` (effective batch size = 16), yielding 63 batches per epoch on 500 samples.

In [7]:
from tqdm.auto import tqdm

train_dataloader = DataLoader(processed_dataset["train"], batch_size=8, shuffle=True)
GRAD_ACCUM_STEPS = 2  # Effective batch size = 16

## Step 8: Phase 1 — Researcher Single-Agent Warm-Up

### Objective
Establishes a baseline bottleneck mapping by training only Researcher $M_A$ and the linear codebook head while keeping Reader $M_B$ frozen (`requires_grad = False`).

$$\mathcal{L}_{\text{Phase 1}} = \mathcal{L}_{\text{answer}}$$

In [ ]:
print("Phase 1: Single-Agent Warm-Up")
# Freeze Reader
for name, param in system.reader.named_parameters():
    if "lora_" in name:
        param.requires_grad = False

system.train()
E1_EPOCHS = 3

for epoch in range(E1_EPOCHS):
    epoch_loss = 0.0
    optimizer.zero_grad()
    progress_bar = tqdm(train_dataloader, desc=f"Phase 1 - Epoch {epoch+1}/{E1_EPOCHS}")
    
    for step, batch in enumerate(progress_bar):
        inputs = system.tokenizer(
            batch["researcher_input"], 
            padding=True, 
            truncation=True, 
            max_length=384, 
            return_tensors="pt"
        ).to(DEVICE)
        
        targets = system.tokenizer(
            batch["reader_target"], 
            padding=True, 
            truncation=True, 
            max_length=32, 
            return_tensors="pt"
        ).input_ids.to(DEVICE)

        loss_ans, logits, message_soft = system(inputs.input_ids, inputs.attention_mask, target_ids=targets)
        loss = loss_ans / GRAD_ACCUM_STEPS
        loss.backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()

        epoch_loss += loss_ans.item()
        progress_bar.set_postfix({"batch_loss": f"{loss_ans.item():.4f}"})
        
        if step % 50 == 0:
            torch.cuda.empty_cache()

    avg_loss = epoch_loss / len(train_dataloader)
    print(f"Phase 1 - Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

## Step 9: Phase 2 — Joint Bottlenecked Supervised Fine-Tuning (SFT)

### Objective
Unfreezes Reader LoRA parameters (`requires_grad = True` on `lora_` layers) and trains both models end-to-end using the full multi-component loss function:

$$\mathcal{L}_{\text{SFT}} = \mathcal{L}_{\text{answer}} + 0.01 \mathcal{L}_{\text{util}} + 0.05 \mathcal{L}_{\text{sep}}$$

In [ ]:
print("Phase 2: Joint Bottlenecked SFT")
# Unfreeze Reader LoRA Adapters
for name, param in system.reader.named_parameters():
    if "lora_" in name:
        param.requires_grad = True

torch.cuda.empty_cache()
E2_EPOCHS = 3

for epoch in range(E2_EPOCHS):
    epoch_loss, epoch_util, epoch_sep = 0.0, 0.0, 0.0
    progress_bar = tqdm(train_dataloader, desc=f"Phase 2 - Epoch {epoch+1}/{E2_EPOCHS}")

    for step, batch in enumerate(progress_bar):
        inputs = system.tokenizer(
            batch["researcher_input"], padding=True, truncation=True, max_length=512, return_tensors="pt"
        ).to(DEVICE)
        targets = system.tokenizer(
            batch["reader_target"], padding=True, truncation=True, max_length=64, return_tensors="pt"
        ).input_ids.to(DEVICE)

        optimizer.zero_grad()
        loss_ans, logits, message_soft = system(inputs.input_ids, inputs.attention_mask, target_ids=targets)
        total_loss, loss_u, loss_s = compute_joint_sft_loss(loss_ans, logits, message_soft)

        total_loss.backward()
        optimizer.step()

        epoch_loss += total_loss.item()
        epoch_util += loss_u.item()
        epoch_sep += loss_s.item()

        progress_bar.set_postfix({
            "total": f"{total_loss.item():.4f}", 
            "util": f"{loss_u.item():.4f}", 
            "sep": f"{loss_s.item():.4f}"
        })
        
        if step % 10 == 0:
            torch.cuda.empty_cache()

    print(f"Phase 2 - Epoch {epoch+1} Loss: {epoch_loss/len(train_dataloader):.4f} | Util: {epoch_util/len(train_dataloader):.4f}")

## Step 10: Phase 3 — Self-Distillation Under Channel Noise

### Objective
Freezes Researcher $M_A$ and trains Reader $M_B$ under $15\%$ synthetic token noise (`noise_rate = 0.15`).

### Noise Channel Model
Given predicted logits $Z_{k,v}$, noise mask $\eta \sim \text{Bernoulli}(0.15)$ randomly replaces uncorrupted logits with random normal noise $\epsilon \sim \mathcal{N}(0, I)$:

$$\hat{Z}_{k,v} = (1 - \eta) Z_{k,v} + \eta \epsilon_{k,v}$$

This forces Reader $M_B$ to develop error correction and decode underlying task semantics robustly.

In [ ]:
print("Phase 3: Self-Distillation")
# Freeze Researcher
for name, param in system.researcher.named_parameters():
    if "lora_" in name:
        param.requires_grad = False

torch.cuda.empty_cache()
E3_EPOCHS = 2

for epoch in range(E3_EPOCHS):
    epoch_loss = 0.0
    progress_bar = tqdm(train_dataloader, desc=f"Phase 3 - Epoch {epoch+1}/{E3_EPOCHS}")

    for step, batch in enumerate(progress_bar):
        inputs = system.tokenizer(
            batch["researcher_input"], padding=True, truncation=True, max_length=512, return_tensors="pt"
        ).to(DEVICE)
        targets = system.tokenizer(
            batch["reader_target"], padding=True, truncation=True, max_length=64, return_tensors="pt"
        ).input_ids.to(DEVICE)

        optimizer.zero_grad()
        # Train Reader under 15% channel noise
        loss_distill, _, _ = system(inputs.input_ids, inputs.attention_mask, target_ids=targets, noise_rate=0.15)
        loss_distill.backward()
        optimizer.step()

        epoch_loss += loss_distill.item()
        progress_bar.set_postfix({"distill_loss": f"{loss_distill.item():.4f}"})

        if step % 10 == 0:
            torch.cuda.empty_cache()

    print(f"Phase 3 - Epoch {epoch+1} Average Distillation Loss: {epoch_loss/len(train_dataloader):.4f}")

## Step 11: Phase 4 — Joint GRPO Policy Optimization

### Objective
Optimizes discrete communication policy using **Group Relative Policy Optimization (GRPO)** with hard discrete token argmax sampling.

### Reward Formulation
$$R(m, a) = \mathbb{I}[a = a^*] + \beta_1 \cdot \frac{\vert{}m\vert{}_{\text{unique}}}{K} - \beta_2 \cdot \frac{\vert{}m\vert{}}{K_{\text{max}}}$$

- **Accuracy Reward ($\mathbb{I}[a = a^*]$):** $+1.0$ if decoded answer matches ground truth.
- **Diversity Reward ($\beta_1 = 0.1$):** Rewards usage of distinct tokens within the 8-token sequence.
- **Compression Penalty ($\beta_2 = 0.05$):** Penalizes unnecessary message length usage.
- **Policy Gradient Loss:** Uses REINFORCE policy loss $\mathcal{L}_{\text{policy}} = - \log P(m \vert{} x) \cdot R(m, a)$.

In [ ]:
print("Phase 4: Joint GRPO Policy Optimization")
for name, param in system.named_parameters():
    if "lora_" in name or "codebook_" in name:
        param.requires_grad = True

torch.cuda.empty_cache()
GRPO_STEPS = 100
data_iter = iter(train_dataloader)

for step in range(GRPO_STEPS):
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(train_dataloader)
        batch = next(data_iter)

    inputs = system.tokenizer(
        batch["researcher_input"], padding=True, truncation=True, max_length=512, return_tensors="pt"
    ).to(DEVICE)
    targets = system.tokenizer(
        batch["reader_target"], padding=True, truncation=True, max_length=64, return_tensors="pt"
    ).input_ids.to(DEVICE)

    optimizer.zero_grad()
    logits, message_soft = system(inputs.input_ids, inputs.attention_mask, hard=True)
    sampled_tokens = torch.argmax(logits[0], dim=-1)

    # Policy rollout evaluation
    reward = compute_grpo_reward("Christopher Nolan", "Christopher Nolan", sampled_tokens)
    
    # REINFORCE policy gradient step
    log_probs = F.log_softmax(logits, dim=-1)
    policy_loss = -log_probs.mean() * reward
    policy_loss.backward()
    optimizer.step()

    if step % 10 == 0:
        print(f"GRPO Step {step}/{GRPO_STEPS} | Policy Loss: {policy_loss.item():.4f} | Rollout Reward: {reward:.4f}")
        torch.cuda.empty_cache()

## Step 12: Artifact Persistence & Local Saving

Saves trained LoRA adapter weights, tokenizer configs, and learned discrete codebook projection head and embedding tables to `/kaggle/working/abstractcomm-qwen3b`.

In [ ]:
print(f"Saving model artifacts locally to {OUTPUT_DIR}...")

# 1. Save LoRA Adapters for Researcher and Reader
system.researcher.save_pretrained(os.path.join(OUTPUT_DIR, "researcher_lora"))
system.reader.save_pretrained(os.path.join(OUTPUT_DIR, "reader_lora"))
system.tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "tokenizer"))

# 2. Save Discrete Codebook Bottleneck Head & Embeddings
codebook_state = {
    "codebook_head": system.codebook_head.state_dict(),
    "codebook_embeddings": system.codebook_embeddings.state_dict()
}
torch.save(codebook_state, os.path.join(OUTPUT_DIR, "codebook_weights.pt"))
print("Model artifacts saved locally.")

## Step 13: Hugging Face Model Hub Upload

Pushes all trained LoRA adapter weights and codebook artifacts directly to your Hugging Face Hub repository.

In [ ]:
HF_REPO_NAME = ""  # Replace with your HF username/repo

try:
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_NAME, exist_ok=True)
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HF_REPO_NAME,
        commit_message="Upload AbstractComm Qwen 3B communication protocol adapters and codebook"
    )
    print(f"Successfully uploaded model artifacts to Hugging Face: https://huggingface.co/{HF_REPO_NAME}")
except Exception as e:
    print(f"Could not push to Hugging Face Hub automatically. Verify HF_TOKEN and HF_REPO_NAME. Error: {e}")